# Colab: train & evaluate CLIPSeg for Drywall QA
**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

Runs are written to Google Drive, so checkpoints survive a disconnect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code + data
For a private repo use `https://<TOKEN>@github.com/...` (GitHub → Settings → Developer settings → fine-grained token, read-only).

In [ ]:
REPO = "https://github.com/VVISHUS/ORIGIN_Prompted_Segmentation.git"
!git clone -q --depth 1 $REPO /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

## 2. Persist runs on Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/drywall_runs"
CFG = "configs/clipseg_ft.yaml"
import os; os.makedirs(RUNS, exist_ok=True)

## 3. Prepare masks + augmentation-aware split (≈1–2 min)

In [ ]:
!python prepare_data.py --config $CFG

## 4. Zero-shot baseline (no training)

In [ ]:
!python evaluate.py --config $CFG --set output_dir={RUNS}

## 5. Fine-tune
Override anything with `--set`, e.g. `train.epochs=30 optim.lr=5e-4`.

In [ ]:
!python train.py --config $CFG --set output_dir={RUNS}

## 6. Evaluate on test + write prediction masks

In [ ]:
import glob
RUN = sorted(glob.glob(f"{RUNS}/clipseg_ft-*"))[-1]   # latest run
print(RUN)
!python evaluate.py --run "{RUN}"
!python predict.py --run "{RUN}" --split test

## 7. Look at the results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{RUN}/eval_test/results.md").read()))
for f in sorted(glob.glob(f"{RUN}/eval_test/examples_*.png")):
    display(Image(f))

## 8. (optional) Download everything as a zip

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/" + os.path.basename(RUN), "zip", RUN)
files.download(zip_path)